# Week 2 Day 1: OpenAI-Compatible APIs with local Ollama
Today we will learn how to use the official OpenAI python library to interact with local Ollama models.

## Why this is important:
In professional AI engineering jobs, you want to write code that is modular. By using the standard OpenAI client, you can switch from a local model (for cheap testing) to a cloud model (for production) by changing only a single line of code (the base URL address)!

In [1]:
# Import the standard OpenAI client library
from openai import OpenAI # type: ignore 

print("OpenAI client library imported successfully!")

OpenAI client library imported successfully!


In [2]:
# Initialize the client pointing to your local Ollama server
# We use a dummy API key "ollama" because local models do not require authentication keys
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

print("Local OpenAI-compatible client initialized!")

Local OpenAI-compatible client initialized!


In [6]:
# 1. Call the local model using the standard OpenAI client
response = client.chat.completions.create(
    model="llama3.2:1b",
    messages=[
        {
            "role": "user",
            "content": "Explain what a neural network is in exactly one sentence."
        }
    ]
)
# 2. Extract the text response (OpenAI SDK style)
reply = response.choices[0].message.content

# 3. Print the output
print(reply)

A neural network is a computer system inspired by the structure and function of the human brain, consisting of interconnected nodes or "neurons" that process and transmit information through weighted connections.


In [7]:
response

ChatCompletion(id='chatcmpl-867', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='A neural network is a computer system inspired by the structure and function of the human brain, consisting of interconnected nodes or "neurons" that process and transmit information through weighted connections.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1789902262, model='llama3.2:1b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=37, prompt_tokens=37, total_tokens=74, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=20)))

# Try With Groq API

Option 1 - With OpenAI Client

In [8]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url="https://api.Groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)

print("Groq OpenAI-compatible client initialized!")

Groq OpenAI-compatible client initialized!


In [ ]:
# 1. Call the model using the standard OpenAI client
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "Explain what a neural network is in exactly one sentence."
        }
    ]
)
# 2. Extract the text response (OpenAI SDK style)
reply = response.choices[0].message.content

# 3. Print the output
print(reply)

A neural network is a computational model composed of interconnected layers of artificial neurons that learn to recognize patterns and make predictions by adjusting weighted connections through training on data.


Option 2 - With Groq Client

In [11]:
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# 1. Call the model using the standard Groq client
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "Explain what a neural network is in exactly one sentence."
        }
    ]
)
# 2. Extract the text response (OpenAI SDK style)
reply = response.choices[0].message.content

# 3. Print the output
print(reply)

A neural network is a computational model composed of interconnected layers of artificial neurons that learn to recognize patterns and make predictions by adjusting weighted connections through training on data.


## Streaming Responses (Real-Time Output)
By default, the API waits until the model generates the entire response before returning it. 

To enable streaming:
1. We set `stream=True` in the API call.
2. The response returns an iterator object (a stream of chunks).
3. We loop through the chunks as they arrive.
4. For each chunk, we extract the delta text: `chunk.choices[0].delta.content`
5. We print it with `end=""` and `flush=True` to show it on the same line in real-time.

flush=True — Show the text immediately
Python sometimes temporarily stores printed output in a buffer before displaying it.

In [19]:
# 1. Start a streaming request by adding stream=True
stream = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "Explain what a neural network is in exactly 5 Line"
        }
    ],
    stream=True # Tell the model to stream responses in real-time!
)

print("Model is writing:")
print("-" * 50)

# 2. Loop through the chunks as they arrive from the local server
for chunk in stream:
    # 3. Check if there is new text content in this chunk
    chunk_text = chunk.choices[0].delta.content
    
    if chunk_text is not None:
        # Print it immediately without a new line
        print(chunk_text, end="", flush=True)

print("\n" + "-" * 50)

Model is writing:
--------------------------------------------------
A neural network is a computational model inspired by the brain’s structure.  
It consists of layers of interconnected “neurons” that process data.  
Each neuron applies a weighted sum and a non‑linear activation function.  
During training, the network adjusts its weights to minimize prediction error.  
After learning, it can recognize patterns, make predictions, or generate new data.
--------------------------------------------------
